In [1]:
!pip install pyspark

In [2]:
from google.colab import files
uploaded = files.upload()

Saving attendance_30days_with_breaks.csv to attendance_30days_with_breaks.csv


In [3]:
import os
print(os.listdir("/content"))

['.config', 'attendance_30days_with_breaks.csv', 'sample_data']


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Attendance Analysis") \
    .getOrCreate()

# Load the attendance CSV file
df = spark.read.csv(
    "/content/attendance_30days_with_breaks.csv",
    header=True,
    inferSchema=True
)

In [6]:
df.show(5)

+-----------+-------------+----------+---------------+-------------------+-------------------+-------+-------------+
|employee_id|employee_name|department|attendance_date|           clock_in|          clock_out| status|break_minutes|
+-----------+-------------+----------+---------------+-------------------+-------------------+-------+-------------+
|          2| Priya Sharma|        HR|     2026-09-01|2026-09-01 08:10:00|2026-09-01 16:15:00|Present|           60|
|          2| Priya Sharma|        HR|     2026-09-02|2026-09-02 08:30:00|2026-09-02 16:45:00|Present|           30|
|          2| Priya Sharma|        HR|     2026-09-03|2026-09-03 08:05:00|2026-09-03 16:00:00|Present|           60|
|          2| Priya Sharma|        HR|     2026-09-04|2026-09-04 09:05:00|2026-09-04 17:30:00|Present|           60|
|          2| Priya Sharma|        HR|     2026-09-07|2026-09-07 08:05:00|2026-09-07 18:45:00|Present|           30|
+-----------+-------------+----------+---------------+----------

In [7]:
df.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- attendance_date: date (nullable = true)
 |-- clock_in: timestamp (nullable = true)
 |-- clock_out: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- break_minutes: integer (nullable = true)



In [8]:
print("Total attendance records:", df.count())

Total attendance records: 154


In [9]:
from pyspark.sql import functions as F
# Display the attendance status counts
print("===== ATTENDANCE STATUS SUMMARY =====")
df.groupBy("status").count().show()
# Filter absent employees
absent_employees = df.filter(
    F.lower(F.col("status")) == "absent"
)
print("===== ABSENT EMPLOYEE RECORDS =====")
absent_employees.select(
    "employee_id",
    "employee_name",
    "attendance_date",
    "status"
).show(50, truncate=False)
# Count total absent records
print("Total absent records:", absent_employees.count())

===== ATTENDANCE STATUS SUMMARY =====
+-------+-----+
| status|count|
+-------+-----+
|Present|  140|
| Absent|   14|
+-------+-----+

===== ABSENT EMPLOYEE RECORDS =====
+-----------+-------------+---------------+------+
|employee_id|employee_name|attendance_date|status|
+-----------+-------------+---------------+------+
|5          |Rahul M      |2026-09-04     |Absent|
|5          |Rahul M      |2026-09-16     |Absent|
|6          |Manoj Kumar  |2026-09-18     |Absent|
|6          |Manoj Kumar  |2026-09-29     |Absent|
|7          |Varun Verma  |2026-09-14     |Absent|
|7          |Varun Verma  |2026-09-30     |Absent|
|1          |Arun Kumar   |2026-09-01     |Absent|
|1          |Arun Kumar   |2026-09-21     |Absent|
|1          |Arun Kumar   |2026-09-23     |Absent|
|3          |Karthik Raj  |2026-09-10     |Absent|
|3          |Karthik Raj  |2026-09-18     |Absent|
|3          |Karthik Raj  |2026-09-28     |Absent|
|4          |Divya S      |2026-09-04     |Absent|
|4          |

In [10]:
from pyspark.sql import functions as F
# Convert clock-in time to a timestamp
df = df.withColumn(
    "clock_in",
    F.to_timestamp("clock_in")
)
# Identify late logins after 9:15 AM
late_logins = df.filter(
    (F.lower(F.col("status")) == "present") &
    (F.date_format("clock_in", "HH:mm:ss") > F.lit("09:15:00"))
)
print("===== LATE LOGIN REPORT =====")
late_logins.select(
    "employee_id",
    "employee_name",
    "attendance_date",
    "clock_in",
    "status"
).show(50, truncate=False)
print("Total late login records:", late_logins.count())

===== LATE LOGIN REPORT =====
+-----------+-------------+---------------+-------------------+-------+
|employee_id|employee_name|attendance_date|clock_in           |status |
+-----------+-------------+---------------+-------------------+-------+
|2          |Priya Sharma |2026-09-10     |2026-09-10 09:30:00|Present|
|2          |Priya Sharma |2026-09-11     |2026-09-11 09:30:00|Present|
|2          |Priya Sharma |2026-09-25     |2026-09-25 09:30:00|Present|
|2          |Priya Sharma |2026-09-30     |2026-09-30 09:30:00|Present|
|5          |Rahul M      |2026-09-01     |2026-09-01 09:30:00|Present|
|5          |Rahul M      |2026-09-07     |2026-09-07 09:30:00|Present|
|5          |Rahul M      |2026-09-08     |2026-09-08 09:30:00|Present|
|6          |Manoj Kumar  |2026-09-17     |2026-09-17 09:30:00|Present|
|7          |Varun Verma  |2026-09-07     |2026-09-07 09:30:00|Present|
|7          |Varun Verma  |2026-09-11     |2026-09-11 09:30:00|Present|
|1          |Arun Kumar   |2026-09

In [12]:
from pyspark.sql import functions as F
# Convert clock-in and clock-out columns to timestamps
df = df.withColumn(
    "clock_in",
    F.to_timestamp("clock_in")
).withColumn(
    "clock_out",
    F.to_timestamp("clock_out")
)
# Calculate total shift hours
df = df.withColumn(
    "working_hours",
    (F.unix_timestamp("clock_out") -
     F.unix_timestamp("clock_in")) / 3600
)
# Calculate department-wise average working hours
department_hours = df.filter(
    F.lower(F.col("status")) == "present"
).groupBy(
    "department"
).agg(
    F.round(F.avg("working_hours"), 2).alias("average_working_hours"),
    F.count("*").alias("attendance_records")
)
print("===== DEPARTMENT-WISE WORKING HOURS =====")
department_hours.show(truncate=False)

===== DEPARTMENT-WISE WORKING HOURS =====
+----------+---------------------+------------------+
|department|average_working_hours|attendance_records|
+----------+---------------------+------------------+
|HR        |8.52                 |42                |
|Finance   |8.77                 |40                |
|IT        |8.51                 |58                |
+----------+---------------------+------------------+



In [13]:
from google.colab import files
uploaded_tasks = files.upload()

Saving tasks_sample.csv to tasks_sample.csv


In [14]:
from pyspark.sql import functions as F
# Load the task dataset
tasks_df = spark.read.csv(
    "/content/tasks_sample.csv",
    header=True,
    inferSchema=True
)
print("===== TASK DATA =====")
tasks_df.show(5, truncate=False)
print("===== TASK DATA SCHEMA =====")
tasks_df.printSchema()

===== TASK DATA =====
+-------+-----------+-------------+----------+---------------+----------+---------------+-----------+
|task_id|employee_id|employee_name|department|task_name      |task_date |tasks_completed|task_status|
+-------+-----------+-------------+----------+---------------+----------+---------------+-----------+
|1      |2          |Priya Sharma |HR        |Documentation  |2026-09-16|0              |Pending    |
|2      |2          |Priya Sharma |HR        |Data cleaning  |2026-09-11|1              |Completed  |
|3      |2          |Priya Sharma |HR        |Data validation|2026-09-28|0              |Pending    |
|4      |2          |Priya Sharma |HR        |Documentation  |2026-09-27|4              |Completed  |
|5      |2          |Priya Sharma |HR        |Monthly report |2026-09-03|2              |Completed  |
+-------+-----------+-------------+----------+---------------+----------+---------------+-----------+
only showing top 5 rows
===== TASK DATA SCHEMA =====
root
 |

In [15]:
from google.colab import files
uploaded_tasks = files.upload()

Saving employees_from_mysql.csv to employees_from_mysql.csv


In [17]:
# Load employee details
employees_df = spark.read.csv(
    "/content/employees_from_mysql.csv",
    header=True,
    inferSchema=True
)
# Avoid duplicate/ambiguous columns like 'employee_name' and 'department'
# by only joining columns from employees_df that aren't already in tasks_df (except the join key)
employee_tasks = tasks_df.join(
    employees_df.select("employee_id"),
    on="employee_id",
    how="left"
)
# Summarize tasks by department using the department column from tasks_df
department_productivity = employee_tasks.groupBy(
    "department"
).agg(
    F.count("*").alias("total_task_records"),
    F.sum(
        F.when(F.col("task_status") == "Completed", 1).otherwise(0)
    ).alias("completed_task_records"),
    F.sum("tasks_completed").alias("total_tasks_completed")
)
print("===== DEPARTMENT-WISE PRODUCTIVITY =====")
department_productivity.show(truncate=False)

===== DEPARTMENT-WISE PRODUCTIVITY =====
+----------+------------------+----------------------+---------------------+
|department|total_task_records|completed_task_records|total_tasks_completed|
+----------+------------------+----------------------+---------------------+
|HR        |10                |6                     |18                   |
|Finance   |9                 |4                     |11                   |
|IT        |12                |6                     |19                   |
+----------+------------------+----------------------+---------------------+



In [18]:
from pyspark.sql import functions as F
# Identify absent records
absent_report = df.filter(
    F.lower(F.col("status")) == "absent"
)
# Save absent employee records
absent_report.write.mode("overwrite").option(
    "header", True
).csv("/content/absent_employees_report")
print("Absent employee report saved successfully!")
# Identify present records with clock-in after 9:15 AM
late_report = df.filter(
    (F.lower(F.col("status")) == "present") &
    (F.date_format("clock_in", "HH:mm:ss") > "09:15:00")
)
# Save late login records
late_report.write.mode("overwrite").option(
    "header", True
).csv("/content/late_login_report")
print("Late login report saved successfully!")

Absent employee report saved successfully!
Late login report saved successfully!


In [19]:
# Save department-wise working hours
department_hours.write.mode("overwrite").option(
    "header", True
).csv("/content/department_working_hours_report")

# Save department-wise productivity
department_productivity.write.mode("overwrite").option(
    "header", True
).csv("/content/department_productivity_report")

print("Department working-hours report saved!")
print("Department productivity report saved!")

Department working-hours report saved!
Department productivity report saved!


In [20]:
import shutil
shutil.make_archive(
    "/content/week3_pyspark_reports",
    "zip",
    "/content",
    "absent_employees_report"
)
print("Absent employee report folder compressed successfully!")

Absent employee report folder compressed successfully!


In [21]:
from google.colab import files
files.download("/content/week3_pyspark_reports.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>